# Newsroom Article Tagging System
## Information Retrieval Assignment



# STUDENT NAME 
## ARUNKUMAR K A - 2024AC05045
## GOWRISHANKAR S - 2024AC05046
## KODHANDAN S - 2024AC05203
## KUMARESH BABU A - 2024AC05304



**Domain:** Intelligent Newsroom Article Tagging & Routing

**Objective:** Implement and evaluate two classifiers (Naïve Bayes and Rocchio) for categorizing news articles into Sports, Politics, and Technology.

**Dataset:** 135+ news articles in **mixed formats**
- Clear-category articles
- Ambiguous cross-category articles
- Articles designed to test classifier robustness

**File Naming Convention:** `article_N_category.extension`
- Example: `article_1_sports.pdf`, `article_2_politics.docx`, `article_3_technology.csv`

**File Formats:** 
- PDF files
- DOCX files
- CSV files
- TXT files

**Key Requirements:**
- Manual implementation of TF-IDF with cosine normalization
- Naïve Bayes classifier (from scratch)
- Rocchio classifier (from scratch)
- 75/25 train-test split
- Evaluation metrics: Accuracy, Precision, Recall, F1-Score

## 1. Import Required Libraries

We use built-in Python libraries for preprocessing only (as permitted by assignment requirements).
Libraries for file format extraction (PyPDF2, python-docx, csv) are used for data loading only.

In [300]:
import os
import re
import math
import random
import csv
from collections import defaultdict, Counter
from typing import List, Dict, Tuple, Set

# For preprocessing only
import string

# For file format extraction (data loading only)
import PyPDF2
from docx import Document

print("Libraries imported successfully!")
print("\nNote: All classifier and TF-IDF implementations are manual (no ML libraries used)")
print("File format libraries (PyPDF2, python-docx, csv) used only for data extraction")

Libraries imported successfully!

Note: All classifier and TF-IDF implementations are manual (no ML libraries used)
File format libraries (PyPDF2, python-docx, csv) used only for data extraction


## 2. Data Loading and Exploration

Load news articles from **multiple file formats** (PDF, DOCX, CSV, TXT) and examine the distribution.

**File Format Support:**
- **PDF**: Extracted using PyPDF2
- **DOCX**: Extracted using python-docx
- **CSV**: Extracted using built-in csv module
- **TXT**: Read directly

In [301]:
def extract_text_from_pdf(filepath):
    """Extract text content from PDF file."""
    text = ""
    try:
        with open(filepath, 'rb') as f:
            pdf_reader = PyPDF2.PdfReader(f)
            for page in pdf_reader.pages:
                text += page.extract_text()
    except Exception as e:
        print(f"Error reading PDF {filepath}: {e}")
    return text

def extract_text_from_docx(filepath):
    """Extract text content from DOCX file."""
    text = ""
    try:
        doc = Document(filepath)
        for para in doc.paragraphs:
            text += para.text + " "
    except Exception as e:
        print(f"Error reading DOCX {filepath}: {e}")
    return text

def extract_text_from_csv(filepath):
    """Extract text content from CSV file."""
    text = ""
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            reader = csv.reader(f)
            for row in reader:
                text += " ".join(row) + " "
    except Exception as e:
        print(f"Error reading CSV {filepath}: {e}")
    return text

def extract_text_from_txt(filepath):
    """Extract text content from TXT file."""
    text = ""
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            text = f.read()
    except Exception as e:
        print(f"Error reading TXT {filepath}: {e}")
    return text

def load_dataset(dataset_path):
    """
    Load all news articles from the dataset directory.
    Supports multiple file formats: PDF, DOCX, CSV, TXT.
    New filename format: article_N_category.extension
    Returns list of (text, category) tuples and format counts.
    """
    documents = []
    format_counts = Counter()
    
    for filename in os.listdir(dataset_path):
        filepath = os.path.join(dataset_path, filename)
        
        # Skip if not a file
        if not os.path.isfile(filepath):
            continue
        
        # Extract category from new filename format (e.g., 'article_1_sports.pdf' -> 'Sports')
        # Format: article_NUMBER_CATEGORY.extension
        category = None
        if filename.startswith('article_') and '_' in filename:
            parts = filename.split('_')
            if len(parts) >= 3:
                # Get category (third part, before extension)
                category_part = parts[2].split('.')[0]  # Remove extension
                if category_part in ['sports', 'politics', 'technology']:
                    category = category_part.capitalize()
        
        if category:
            # Extract text based on file extension
            text = ""
            if filename.endswith('.pdf'):
                text = extract_text_from_pdf(filepath)
                format_counts['PDF'] += 1
            elif filename.endswith('.docx'):
                text = extract_text_from_docx(filepath)
                format_counts['DOCX'] += 1
            elif filename.endswith('.csv'):
                text = extract_text_from_csv(filepath)
                format_counts['CSV'] += 1
            elif filename.endswith('.txt'):
                text = extract_text_from_txt(filepath)
                format_counts['TXT'] += 1
            
            if text.strip():  # Only add if text was successfully extracted
                documents.append((text, category))
    
    return documents, format_counts

# Load dataset
dataset_path = 'news_dataset_mixed'
print("Loading dataset from multiple file formats...")
print("New filename format: article_N_category.extension")
print("="*80)
documents, format_counts = load_dataset(dataset_path)

# Display statistics
print(f"\nTotal documents loaded: {len(documents)}")

print("\nFile Format Distribution:")
for fmt, count in sorted(format_counts.items()):
    print(f"  {fmt}: {count} files")

print("\nCategory Distribution:")
category_counts = Counter([doc[1] for doc in documents])
for category, count in sorted(category_counts.items()):
    print(f"  {category}: {count} articles")

# Display sample document
print("\n" + "="*80)
print("Sample Document:")
print("="*80)
print(f"Category: {documents[0][1]}")
print(f"Text Preview:\n{documents[0][0][:300]}...")
print(f"\nFull document length: {len(documents[0][0])} characters")

Loading dataset from multiple file formats...
New filename format: article_N_category.extension

Total documents loaded: 135

File Format Distribution:
  CSV: 37 files
  DOCX: 38 files
  PDF: 40 files
  TXT: 20 files

Category Distribution:
  Politics: 39 articles
  Sports: 44 articles
  Technology: 52 articles

Sample Document:
Category: Technology
Text Preview:
Technology Article 46
Internet of Things devices proliferate connecting everyday objects to networks. Smart home
technology automates tasks and monitors environments remotely. Privacy and security challenges
accompany increased connectivity and data collection....

Full document length: 261 characters


## 3. Text Preprocessing

Implement preprocessing pipeline: tokenization, lowercasing, stop word removal, and stemming.

**Note:** Built-in libraries are permitted for preprocessing only.

In [302]:
# Define common English stop words
STOP_WORDS = set([
    'a', 'about', 'above', 'after', 'again', 'against', 'all', 'am', 'an', 'and', 'any', 'are', 
    'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 
    'by', 'could', 'did', 'do', 'does', 'doing', 'down', 'during', 'each', 'few', 'for', 'from', 
    'further', 'had', 'has', 'have', 'having', 'he', 'her', 'here', 'hers', 'herself', 'him', 
    'himself', 'his', 'how', 'i', 'if', 'in', 'into', 'is', 'it', 'its', 'itself', 'just', 'me', 
    'might', 'more', 'most', 'must', 'my', 'myself', 'no', 'nor', 'not', 'now', 'of', 'off', 
    'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 'same', 
    'she', 'should', 'so', 'some', 'such', 'than', 'that', 'the', 'their', 'theirs', 'them', 
    'themselves', 'then', 'there', 'these', 'they', 'this', 'those', 'through', 'to', 'too', 
    'under', 'until', 'up', 'very', 'was', 'we', 'were', 'what', 'when', 'where', 'which', 'while', 
    'who', 'whom', 'why', 'will', 'with', 'would', 'you', 'your', 'yours', 'yourself', 'yourselves'
])

def simple_stemmer(word):
    """
    Simple rule-based stemmer (Porter-like stemming rules).
    This is a basic implementation for demonstration.
    """
    # Remove common suffixes
    suffixes = ['ing', 'ed', 'ly', 'es', 's', 'ment', 'ness', 'tion', 'sion']
    
    for suffix in suffixes:
        if word.endswith(suffix) and len(word) > len(suffix) + 2:
            return word[:-len(suffix)]
    
    return word

def preprocess_text(text):
    """
    Preprocess text: tokenization, lowercasing, stop word removal, stemming.
    Returns list of preprocessed tokens.
    """
    # Convert to lowercase
    text = text.lower()
    
    # Remove punctuation and tokenize
    text = re.sub(f'[{re.escape(string.punctuation)}]', ' ', text)
    tokens = text.split()
    
    # Remove stop words and apply stemming
    tokens = [simple_stemmer(token) for token in tokens 
              if token not in STOP_WORDS and len(token) > 2]
    
    return tokens

# Test preprocessing
sample_text = documents[0][0]
preprocessed = preprocess_text(sample_text)

print("Original Text (first 200 chars):")
print(sample_text[:200])
print("\n" + "="*80)
print(f"\nPreprocessed Tokens (first 30):")
print(preprocessed[:30])
print(f"\nTotal tokens after preprocessing: {len(preprocessed)}")

Original Text (first 200 chars):
Technology Article 46
Internet of Things devices proliferate connecting everyday objects to networks. Smart home
technology automates tasks and monitors environments remotely. Privacy and security cha


Preprocessed Tokens (first 30):
['technology', 'article', 'internet', 'thing', 'devic', 'proliferate', 'connect', 'everyday', 'object', 'network', 'smart', 'home', 'technology', 'automat', 'task', 'monitor', 'environment', 'remote', 'privacy', 'security', 'challeng', 'accompany', 'increas', 'connectivity', 'data', 'collec']

Total tokens after preprocessing: 26


In [303]:
# Preprocess all documents
preprocessed_documents = []
for text, category in documents:
    tokens = preprocess_text(text)
    preprocessed_documents.append((tokens, category))

print(f"Preprocessed {len(preprocessed_documents)} documents")
print("\nSample preprocessed document:")
print(f"Category: {preprocessed_documents[0][1]}")
print(f"Token count: {len(preprocessed_documents[0][0])}")
print(f"First 20 tokens: {preprocessed_documents[0][0][:20]}")

Preprocessed 135 documents

Sample preprocessed document:
Category: Technology
Token count: 26
First 20 tokens: ['technology', 'article', 'internet', 'thing', 'devic', 'proliferate', 'connect', 'everyday', 'object', 'network', 'smart', 'home', 'technology', 'automat', 'task', 'monitor', 'environment', 'remote', 'privacy', 'security']


## 4. Train-Test Split (75%-25%)

Split the dataset into training (75%) and testing (25%) sets with stratified sampling.

In [304]:
def train_test_split_stratified(documents, test_size=0.25, random_seed=456):
    """
    Perform stratified train-test split to maintain category proportions.
    Different seed value ensures varied test set composition.
    """
    random.seed(random_seed)
    
    # Group documents by category
    category_docs = defaultdict(list)
    for doc, category in documents:
        category_docs[category].append((doc, category))
    
    train_data = []
    test_data = []
    
    # Split each category
    for category, docs in category_docs.items():
        random.shuffle(docs)
        split_idx = int(len(docs) * (1 - test_size))
        train_data.extend(docs[:split_idx])
        test_data.extend(docs[split_idx:])
    
    # Shuffle final datasets
    random.shuffle(train_data)
    random.shuffle(test_data)
    
    return train_data, test_data
# Perform split (seed=456 provides test set with ambiguous articles)
# Perform split (changed seed to 123 for better test article distribution)
train_data, test_data = train_test_split_stratified(preprocessed_documents, test_size=0.25)

print(f"Training set size: {len(train_data)} documents")
print(f"Test set size: {len(test_data)} documents")
print(f"Split ratio: {len(train_data)/(len(train_data)+len(test_data))*100:.1f}% / {len(test_data)/(len(train_data)+len(test_data))*100:.1f}%")

# Display category distribution in splits
print("\nTraining set distribution:")
train_categories = Counter([doc[1] for doc in train_data])
for cat, count in sorted(train_categories.items()):
    print(f"  {cat}: {count} articles")

print("\nTest set distribution:")
test_categories = Counter([doc[1] for doc in test_data])
for cat, count in sorted(test_categories.items()):
    print(f"  {cat}: {count} articles")

Training set size: 101 documents
Test set size: 34 documents
Split ratio: 74.8% / 25.2%

Training set distribution:
  Politics: 29 articles
  Sports: 33 articles
  Technology: 39 articles

Test set distribution:
  Politics: 10 articles
  Sports: 11 articles
  Technology: 13 articles


## 5. Manual TF-IDF Implementation

Implement TF-IDF calculation from scratch with:
- Term Frequency (TF)
- Inverse Document Frequency (IDF)
- Cosine (L2) normalization

**No external libraries used for TF-IDF computation!**

In [305]:
class TfidfVectorizer:
    """
    Manual TF-IDF Vectorizer implementation with cosine normalization.
    """
    
    def __init__(self):
        self.vocabulary = {}  # term -> index mapping
        self.idf_values = {}  # term -> IDF value
        self.num_documents = 0
    
    def fit(self, documents):
        """
        Build vocabulary and compute IDF values from training documents.
        documents: list of token lists
        """
        self.num_documents = len(documents)
        
        # Build vocabulary
        all_terms = set()
        for doc in documents:
            all_terms.update(doc)
        
        self.vocabulary = {term: idx for idx, term in enumerate(sorted(all_terms))}
        
        # Compute document frequency for each term
        doc_freq = defaultdict(int)
        for doc in documents:
            unique_terms = set(doc)
            for term in unique_terms:
                doc_freq[term] += 1
        
        # Compute IDF: log(N / df_t)
        for term in self.vocabulary:
            df = doc_freq[term]
            self.idf_values[term] = math.log(self.num_documents / df)
        
        print(f"Vocabulary built with {len(self.vocabulary)} unique terms")
        print(f"IDF computed for {len(self.idf_values)} terms")
    
    def transform(self, documents):
        """
        Transform documents to TF-IDF vectors with cosine normalization.
        Returns list of TF-IDF vectors (as dictionaries: term -> tfidf_value)
        """
        tfidf_vectors = []
        
        for doc in documents:
            # Compute term frequency
            tf = Counter(doc)
            
            # Compute TF-IDF
            tfidf_vector = {}
            for term, freq in tf.items():
                if term in self.vocabulary:
                    # TF-IDF = TF * IDF
                    tfidf_vector[term] = freq * self.idf_values[term]
            
            # Apply cosine (L2) normalization
            norm = math.sqrt(sum(val ** 2 for val in tfidf_vector.values()))
            if norm > 0:
                tfidf_vector = {term: val / norm for term, val in tfidf_vector.items()}
            
            tfidf_vectors.append(tfidf_vector)
        
        return tfidf_vectors
    
    def fit_transform(self, documents):
        """
        Fit and transform in one step.
        """
        self.fit(documents)
        return self.transform(documents)

# Initialize and fit vectorizer
vectorizer = TfidfVectorizer()
train_tokens = [doc[0] for doc in train_data]
test_tokens = [doc[0] for doc in test_data]

print("\nComputing TF-IDF vectors for training data...")
train_tfidf = vectorizer.fit_transform(train_tokens)

print("\nComputing TF-IDF vectors for test data...")
test_tfidf = vectorizer.transform(test_tokens)

print(f"\nTF-IDF transformation complete!")
print(f"Training vectors: {len(train_tfidf)}")
print(f"Test vectors: {len(test_tfidf)}")


Computing TF-IDF vectors for training data...
Vocabulary built with 1409 unique terms
IDF computed for 1409 terms

Computing TF-IDF vectors for test data...

TF-IDF transformation complete!
Training vectors: 101
Test vectors: 34


In [306]:
# Display sample TF-IDF vectors
print("Sample TF-IDF Vector (First Training Document):")
print("="*80)
sample_vector = train_tfidf[0]
print(f"Category: {train_data[0][1]}")
print(f"Vector dimension: {len(sample_vector)} non-zero terms")
print(f"\nTop 10 terms by TF-IDF score:")
top_terms = sorted(sample_vector.items(), key=lambda x: x[1], reverse=True)[:10]
for term, score in top_terms:
    print(f"  {term:20s} : {score:.4f}")

# Verify normalization
norm = math.sqrt(sum(val ** 2 for val in sample_vector.values()))
print(f"\nVector L2 norm (should be ≈1.0): {norm:.6f}")

Sample TF-IDF Vector (First Training Document):
Category: Politics
Vector dimension: 36 non-zero terms

Top 10 terms by TF-IDF score:
  regula               : 0.2949
  speech               : 0.2949
  media                : 0.2708
  social               : 0.2521
  debate               : 0.2126
  regulat              : 0.1935
  liability            : 0.1935
  platform             : 0.1859
  intensifi            : 0.1644
  house                : 0.1644

Vector L2 norm (should be ≈1.0): 1.000000


## 6. Naïve Bayes Classifier Implementation

Implement Multinomial Naïve Bayes classifier from scratch using:
- Prior probabilities: P(class)
- Likelihood: P(term|class) with Laplace smoothing
- Prediction: argmax P(class) * ∏ P(term|class)

In [307]:
class NaiveBayesClassifier:
    """
    Multinomial Naïve Bayes Classifier - Manual Implementation
    """
    
    def __init__(self, alpha=1.0):
        """
        alpha: Laplace smoothing parameter
        """
        self.alpha = alpha
        self.classes = []
        self.prior_probs = {}  # P(class)
        self.term_probs = {}   # P(term|class)
        self.vocabulary = set()
    
    def fit(self, documents, labels):
        """
        Train the Naïve Bayes classifier.
        documents: list of token lists
        labels: list of category labels
        """
        self.classes = sorted(set(labels))
        n_docs = len(documents)
        
        # Build vocabulary
        for doc in documents:
            self.vocabulary.update(doc)
        
        vocab_size = len(self.vocabulary)
        print(f"Vocabulary size: {vocab_size}")
        
        # Compute prior probabilities P(class)
        class_counts = Counter(labels)
        print(f"\nClass counts: {dict(class_counts)}")
        
        for cls in self.classes:
            self.prior_probs[cls] = class_counts[cls] / n_docs
        
        print(f"\nPrior Probabilities P(class):")
        for cls, prob in self.prior_probs.items():
            print(f"  P({cls}) = {prob:.4f}")
        
        # Compute likelihood P(term|class) with Laplace smoothing
        for cls in self.classes:
            # Get all documents for this class
            class_docs = [doc for doc, label in zip(documents, labels) if label == cls]
            
            # Count term frequencies in this class
            term_counts = Counter()
            for doc in class_docs:
                term_counts.update(doc)
            
            total_terms = sum(term_counts.values())
            
            # Compute P(term|class) with Laplace smoothing
            self.term_probs[cls] = {}
            for term in self.vocabulary:
                count = term_counts[term]
                # P(term|class) = (count + alpha) / (total + alpha * vocab_size)
                self.term_probs[cls][term] = (count + self.alpha) / (total_terms + self.alpha * vocab_size)
        
        print(f"\nLikelihood probabilities computed for {len(self.classes)} classes")
    
    def predict(self, documents):
        """
        Predict class labels for documents.
        Returns list of predicted labels.
        """
        predictions = []
        
        for doc in documents:
            # Calculate log probability for each class
            class_scores = {}
            
            for cls in self.classes:
                # Start with log prior
                log_prob = math.log(self.prior_probs[cls])
                
                # Add log likelihood for each term
                for term in doc:
                    if term in self.vocabulary:
                        log_prob += math.log(self.term_probs[cls][term])
                
                class_scores[cls] = log_prob
            
            # Predict class with highest probability
            predicted_class = max(class_scores, key=class_scores.get)
            predictions.append(predicted_class)
        
        return predictions

# Train Naïve Bayes Classifier
print("Training Naïve Bayes Classifier...")
print("="*80)
# Using alpha=0.5 for more sensitivity to discriminative terms (vs standard 1.0)
nb_classifier = NaiveBayesClassifier(alpha=0.5)
train_labels = [doc[1] for doc in train_data]
nb_classifier.fit(train_tokens, train_labels)
print("\nNaïve Bayes training complete!")
print(f"Smoothing parameter (alpha): 0.5")
print(f"Smoothing parameter (alpha): 0.5")

Training Naïve Bayes Classifier...
Vocabulary size: 1409

Class counts: {'Politics': 29, 'Technology': 39, 'Sports': 33}

Prior Probabilities P(class):
  P(Politics) = 0.2871
  P(Sports) = 0.3267
  P(Technology) = 0.3861

Likelihood probabilities computed for 3 classes

Naïve Bayes training complete!
Smoothing parameter (alpha): 0.5
Smoothing parameter (alpha): 0.5


In [308]:
# Make predictions on test data
print("Making predictions on test data...")
nb_predictions = nb_classifier.predict(test_tokens)
test_labels = [doc[1] for doc in test_data]

print(f"\nPredictions complete for {len(nb_predictions)} test documents")
print("\nSample Predictions (first 10):")
print("-" * 60)
for i in range(min(10, len(nb_predictions))):
    actual = test_labels[i]
    predicted = nb_predictions[i]
    status = "✓" if actual == predicted else "✗"
    print(f"{i+1:2d}. Actual: {actual:12s} | Predicted: {predicted:12s} {status}")

Making predictions on test data...

Predictions complete for 34 test documents

Sample Predictions (first 10):
------------------------------------------------------------
 1. Actual: Technology   | Predicted: Technology   ✓
 2. Actual: Sports       | Predicted: Sports       ✓
 3. Actual: Technology   | Predicted: Technology   ✓
 4. Actual: Sports       | Predicted: Technology   ✗
 5. Actual: Sports       | Predicted: Sports       ✓
 6. Actual: Sports       | Predicted: Sports       ✓
 7. Actual: Technology   | Predicted: Technology   ✓
 8. Actual: Technology   | Predicted: Technology   ✓
 9. Actual: Sports       | Predicted: Sports       ✓
10. Actual: Sports       | Predicted: Sports       ✓


## 7. Rocchio Classifier Implementation

Implement Rocchio classifier using:
- TF-IDF vectors with cosine normalization (already computed)
- Centroid calculation for each class
- Cosine similarity for classification

In [309]:
def cosine_similarity(vec1, vec2):
    """
    Compute cosine similarity between two TF-IDF vectors.
    Vectors are dictionaries: {term: tfidf_value}
    """
    # Get common terms
    common_terms = set(vec1.keys()) & set(vec2.keys())
    
    if not common_terms:
        return 0.0
    
    # Compute dot product
    dot_product = sum(vec1[term] * vec2[term] for term in common_terms)
    
    # Compute magnitudes (should be 1.0 if normalized)
    mag1 = math.sqrt(sum(val ** 2 for val in vec1.values()))
    mag2 = math.sqrt(sum(val ** 2 for val in vec2.values()))
    
    if mag1 == 0 or mag2 == 0:
        return 0.0
    
    return dot_product / (mag1 * mag2)


class RocchioClassifier:
    """
    Rocchio Classifier - Manual Implementation using TF-IDF and Cosine Similarity
    """
    
    def __init__(self):
        self.classes = []
        self.centroids = {}  # class -> centroid vector
    
    def fit(self, tfidf_vectors, labels):
        """
        Train Rocchio classifier by computing class centroids.
        tfidf_vectors: list of TF-IDF vectors (dictionaries)
        labels: list of category labels
        """
        self.classes = sorted(set(labels))
        
        print("Computing class centroids...")
        
        # Compute centroid for each class
        for cls in self.classes:
            # Get all vectors for this class
            class_vectors = [vec for vec, label in zip(tfidf_vectors, labels) if label == cls]
            n_docs = len(class_vectors)
            
            # Compute centroid (average of all vectors)
            centroid = defaultdict(float)
            for vec in class_vectors:
                for term, value in vec.items():
                    centroid[term] += value
            
            # Average
            centroid = {term: value / n_docs for term, value in centroid.items()}
            
            # Normalize centroid (L2 normalization)
            norm = math.sqrt(sum(val ** 2 for val in centroid.values()))
            if norm > 0:
                centroid = {term: val / norm for term, val in centroid.items()}
            
            self.centroids[cls] = dict(centroid)
            
            print(f"  {cls}: centroid with {len(centroid)} non-zero terms")
        
        print("\nRocchio training complete!")
    
    def predict(self, tfidf_vectors):
        """
        Predict class labels using cosine similarity to centroids.
        Returns list of predicted labels.
        """
        predictions = []
        
        for vec in tfidf_vectors:
            # Compute similarity to each class centroid
            similarities = {}
            for cls in self.classes:
                similarities[cls] = cosine_similarity(vec, self.centroids[cls])
            
            # Predict class with highest similarity
            predicted_class = max(similarities, key=similarities.get)
            predictions.append(predicted_class)
        
        return predictions

# Train Rocchio Classifier
print("Training Rocchio Classifier...")
print("="*80)
rocchio_classifier = RocchioClassifier()
rocchio_classifier.fit(train_tfidf, train_labels)

print("\n" + "="*80)

Training Rocchio Classifier...
Computing class centroids...
  Politics: centroid with 556 non-zero terms
  Sports: centroid with 628 non-zero terms
  Technology: centroid with 684 non-zero terms

Rocchio training complete!



In [310]:
# Display centroid information
print("Rocchio Classifier Centroid Details:")
print("="*80)

for cls in rocchio_classifier.classes:
    centroid = rocchio_classifier.centroids[cls]
    print(f"\n{cls} Centroid:")
    print(f"  Non-zero terms: {len(centroid)}")
    print(f"  Top 10 terms by TF-IDF weight:")
    top_terms = sorted(centroid.items(), key=lambda x: x[1], reverse=True)[:10]
    for term, weight in top_terms:
        print(f"    {term:20s} : {weight:.4f}")

Rocchio Classifier Centroid Details:

Politics Centroid:
  Non-zero terms: 556
  Top 10 terms by TF-IDF weight:
    politic              : 0.2197
    political            : 0.1988
    govern               : 0.1785
    legisla              : 0.1604
    policy               : 0.1596
    budget               : 0.1593
    public               : 0.1374
    economic             : 0.1334
    propos               : 0.1216
    reform               : 0.1162

Sports Centroid:
  Non-zero terms: 628
  Top 10 terms by TF-IDF weight:
    championship         : 0.2287
    sport                : 0.2159
    team                 : 0.1887
    player               : 0.1686
    athletic             : 0.1410
    tourna               : 0.1400
    competi              : 0.1198
    victory              : 0.1124
    throughout           : 0.1122
    olympic              : 0.1121

Technology Centroid:
  Non-zero terms: 684
  Top 10 terms by TF-IDF weight:
    technology           : 0.2919
    application         

In [311]:
# Make predictions on test data
print("Making predictions on test data...")
rocchio_predictions = rocchio_classifier.predict(test_tfidf)

print(f"\nPredictions complete for {len(rocchio_predictions)} test documents")
print("\nSample Predictions (first 10):")
print("-" * 60)
for i in range(min(10, len(rocchio_predictions))):
    actual = test_labels[i]
    predicted = rocchio_predictions[i]
    status = "✓" if actual == predicted else "✗"
    print(f"{i+1:2d}. Actual: {actual:12s} | Predicted: {predicted:12s} {status}")

Making predictions on test data...

Predictions complete for 34 test documents

Sample Predictions (first 10):
------------------------------------------------------------
 1. Actual: Technology   | Predicted: Technology   ✓
 2. Actual: Sports       | Predicted: Sports       ✓
 3. Actual: Technology   | Predicted: Technology   ✓
 4. Actual: Sports       | Predicted: Technology   ✗
 5. Actual: Sports       | Predicted: Sports       ✓
 6. Actual: Sports       | Predicted: Sports       ✓
 7. Actual: Technology   | Predicted: Technology   ✓
 8. Actual: Technology   | Predicted: Technology   ✓
 9. Actual: Sports       | Predicted: Sports       ✓
10. Actual: Sports       | Predicted: Sports       ✓


## 8. Evaluation Metrics Implementation

Implement evaluation metrics from scratch:
- Accuracy
- Precision (per-class and macro-averaged)
- Recall (per-class and macro-averaged)
- F1-Score (per-class and macro-averaged)

In [312]:
def compute_confusion_matrix(true_labels, predicted_labels, classes):
    """
    Compute confusion matrix.
    Returns dictionary: {(true_class, pred_class): count}
    """
    matrix = defaultdict(int)
    for true, pred in zip(true_labels, predicted_labels):
        matrix[(true, pred)] += 1
    return matrix


def compute_metrics(true_labels, predicted_labels, classes):
    """
    Compute all evaluation metrics.
    Returns dictionary with accuracy, per-class and macro-averaged metrics.
    """
    # Confusion matrix
    conf_matrix = compute_confusion_matrix(true_labels, predicted_labels, classes)
    
    # Accuracy
    correct = sum(1 for true, pred in zip(true_labels, predicted_labels) if true == pred)
    accuracy = correct / len(true_labels)
    
    # Per-class metrics
    class_metrics = {}
    
    for cls in classes:
        # True Positives
        tp = conf_matrix[(cls, cls)]
        
        # False Positives
        fp = sum(conf_matrix[(other, cls)] for other in classes if other != cls)
        
        # False Negatives
        fn = sum(conf_matrix[(cls, other)] for other in classes if other != cls)
        
        # True Negatives (not commonly used in multi-class, but for completeness)
        tn = sum(conf_matrix[(other1, other2)] 
                for other1 in classes for other2 in classes 
                if other1 != cls and other2 != cls)
        
        # Precision
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        
        # Recall
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        
        # F1-Score
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        
        class_metrics[cls] = {
            'precision': precision,
            'recall': recall,
            'f1_score': f1,
            'support': tp + fn  # Total actual instances of this class
        }
    
    # Macro-averaged metrics
    macro_precision = sum(m['precision'] for m in class_metrics.values()) / len(classes)
    macro_recall = sum(m['recall'] for m in class_metrics.values()) / len(classes)
    macro_f1 = sum(m['f1_score'] for m in class_metrics.values()) / len(classes)
    
    return {
        'accuracy': accuracy,
        'class_metrics': class_metrics,
        'macro_precision': macro_precision,
        'macro_recall': macro_recall,
        'macro_f1': macro_f1,
        'confusion_matrix': conf_matrix
    }


def display_metrics(metrics, classifier_name):
    """
    Display evaluation metrics in a formatted way.
    """
    print(f"\n{'='*80}")
    print(f"{classifier_name} - Evaluation Metrics")
    print(f"{'='*80}")
    
    print(f"\nOverall Accuracy: {metrics['accuracy']:.4f} ({metrics['accuracy']*100:.2f}%)")
    
    print(f"\nPer-Class Metrics:")
    print("-" * 80)
    print(f"{'Class':<15} {'Precision':<12} {'Recall':<12} {'F1-Score':<12} {'Support':<10}")
    print("-" * 80)
    
    for cls, m in sorted(metrics['class_metrics'].items()):
        print(f"{cls:<15} {m['precision']:<12.4f} {m['recall']:<12.4f} {m['f1_score']:<12.4f} {m['support']:<10}")
    
    print("-" * 80)
    print(f"{'Macro Avg':<15} {metrics['macro_precision']:<12.4f} {metrics['macro_recall']:<12.4f} {metrics['macro_f1']:<12.4f}")
    print("-" * 80)
    
    # Confusion Matrix
    print(f"\nConfusion Matrix:")
    print("-" * 60)
    classes = sorted(metrics['class_metrics'].keys())
    print(f"{'Actual \\ Pred':<15}", end="")
    for cls in classes:
        print(f"{cls:<15}", end="")
    print()
    print("-" * 60)
    
    for true_cls in classes:
        print(f"{true_cls:<15}", end="")
        for pred_cls in classes:
            count = metrics['confusion_matrix'][(true_cls, pred_cls)]
            print(f"{count:<15}", end="")
        print()
    print("="*80)

print("Evaluation metrics functions defined successfully!")

Evaluation metrics functions defined successfully!


## 9. Naïve Bayes Classifier Evaluation

In [313]:
# Evaluate Naïve Bayes Classifier
classes = sorted(set(test_labels))
nb_metrics = compute_metrics(test_labels, nb_predictions, classes)
display_metrics(nb_metrics, "Naïve Bayes Classifier")


Naïve Bayes Classifier - Evaluation Metrics

Overall Accuracy: 0.9412 (94.12%)

Per-Class Metrics:
--------------------------------------------------------------------------------
Class           Precision    Recall       F1-Score     Support   
--------------------------------------------------------------------------------
Politics        0.9091       1.0000       0.9524       10        
Sports          1.0000       0.9091       0.9524       11        
Technology      0.9231       0.9231       0.9231       13        
--------------------------------------------------------------------------------
Macro Avg       0.9441       0.9441       0.9426      
--------------------------------------------------------------------------------

Confusion Matrix:
------------------------------------------------------------
Actual \ Pred  Politics       Sports         Technology     
------------------------------------------------------------
Politics       10             0              0         

## 10. Rocchio Classifier Evaluation

In [314]:
# Evaluate Rocchio Classifier
rocchio_metrics = compute_metrics(test_labels, rocchio_predictions, classes)
display_metrics(rocchio_metrics, "Rocchio Classifier")


Rocchio Classifier - Evaluation Metrics

Overall Accuracy: 0.9118 (91.18%)

Per-Class Metrics:
--------------------------------------------------------------------------------
Class           Precision    Recall       F1-Score     Support   
--------------------------------------------------------------------------------
Politics        0.9091       1.0000       0.9524       10        
Sports          0.9091       0.9091       0.9091       11        
Technology      0.9167       0.8462       0.8800       13        
--------------------------------------------------------------------------------
Macro Avg       0.9116       0.9184       0.9138      
--------------------------------------------------------------------------------

Confusion Matrix:
------------------------------------------------------------
Actual \ Pred  Politics       Sports         Technology     
------------------------------------------------------------
Politics       10             0              0             

## 11. Comparative Performance Analysis

In [315]:
print("\n" + "="*80)
print("COMPARATIVE PERFORMANCE ANALYSIS")
print("="*80)

# Summary comparison table
print("\nOverall Performance Comparison:")
print("-" * 80)
print(f"{'Metric':<25} {'Naïve Bayes':<20} {'Rocchio':<20} {'Better':<15}")
print("-" * 80)

metrics_to_compare = [
    ('Accuracy', 'accuracy'),
    ('Macro Precision', 'macro_precision'),
    ('Macro Recall', 'macro_recall'),
    ('Macro F1-Score', 'macro_f1')
]

nb_wins = 0
rocchio_wins = 0

for metric_name, metric_key in metrics_to_compare:
    nb_val = nb_metrics[metric_key]
    rocchio_val = rocchio_metrics[metric_key]
    
    if nb_val > rocchio_val:
        better = "Naïve Bayes"
        nb_wins += 1
    elif rocchio_val > nb_val:
        better = "Rocchio"
        rocchio_wins += 1
    else:
        better = "Tie"
    
    print(f"{metric_name:<25} {nb_val:<20.4f} {rocchio_val:<20.4f} {better:<15}")

print("-" * 80)

# Per-class comparison
print("\nPer-Class F1-Score Comparison:")
print("-" * 70)
print(f"{'Class':<15} {'Naïve Bayes':<20} {'Rocchio':<20} {'Better':<15}")
print("-" * 70)

for cls in classes:
    nb_f1 = nb_metrics['class_metrics'][cls]['f1_score']
    rocchio_f1 = rocchio_metrics['class_metrics'][cls]['f1_score']
    
    if nb_f1 > rocchio_f1:
        better = "Naïve Bayes"
    elif rocchio_f1 > nb_f1:
        better = "Rocchio"
    else:
        better = "Tie"
    
    print(f"{cls:<15} {nb_f1:<20.4f} {rocchio_f1:<20.4f} {better:<15}")

print("-" * 70)

# Final recommendation
print("\n" + "="*80)
print("RECOMMENDATION FOR EDITORIAL TAGGING")
print("="*80)

if nb_metrics['accuracy'] > rocchio_metrics['accuracy']:
    better_classifier = "Naïve Bayes"
    accuracy_diff = (nb_metrics['accuracy'] - rocchio_metrics['accuracy']) * 100
elif rocchio_metrics['accuracy'] > nb_metrics['accuracy']:
    better_classifier = "Rocchio"
    accuracy_diff = (rocchio_metrics['accuracy'] - nb_metrics['accuracy']) * 100
else:
    better_classifier = "Both (Tie)"
    accuracy_diff = 0

print(f"\nRecommended Classifier: {better_classifier}")

if better_classifier != "Both (Tie)":
    print(f"\nAccuracy Advantage: {accuracy_diff:.2f}%")

print("\nJustification:")
print("-" * 80)

if nb_metrics['accuracy'] > rocchio_metrics['accuracy']:
    print("""The Naïve Bayes classifier demonstrates superior performance for this news 
article tagging task. Key advantages include:

1. Higher overall accuracy in categorizing articles across all three categories
2. Better handling of the probabilistic nature of text classification
3. Effective use of term frequencies with Laplace smoothing to handle unseen terms
4. Strong performance even with limited training data

The Naïve Bayes classifier is well-suited for newsroom editorial routing as it
provides reliable classification with interpretable probability estimates.""")
elif rocchio_metrics['accuracy'] > nb_metrics['accuracy']:
    print("""The Rocchio classifier demonstrates superior performance for this news 
article tagging task. Key advantages include:

1. Higher overall accuracy in categorizing articles across all three categories
2. Effective use of TF-IDF vectors to capture term importance
3. Cosine similarity provides robust distance measurement in high-dimensional space
4. Centroid-based approach works well when classes have coherent topic clusters

The Rocchio classifier is well-suited for newsroom editorial routing as it
leverages semantic similarity effectively for document classification.""")
else:
    print("""Both classifiers show comparable performance. The choice may depend on:

- Computational requirements (Rocchio is generally faster at prediction)
- Interpretability needs (Naïve Bayes provides probability scores)
- Update frequency (Naïve Bayes easier to update incrementally)

For a production newsroom system, either classifier would be suitable.""")

print("="*80)


COMPARATIVE PERFORMANCE ANALYSIS

Overall Performance Comparison:
--------------------------------------------------------------------------------
Metric                    Naïve Bayes          Rocchio              Better         
--------------------------------------------------------------------------------
Accuracy                  0.9412               0.9118               Naïve Bayes    
Macro Precision           0.9441               0.9116               Naïve Bayes    
Macro Recall              0.9441               0.9184               Naïve Bayes    
Macro F1-Score            0.9426               0.9138               Naïve Bayes    
--------------------------------------------------------------------------------

Per-Class F1-Score Comparison:
----------------------------------------------------------------------
Class           Naïve Bayes          Rocchio              Better         
----------------------------------------------------------------------
Politics        0.9524

## 12. Summary and Conclusion

In [316]:
print("\n" + "="*80)
print("ASSIGNMENT SUMMARY")
print("="*80)

print("\n✓ Dataset: 60 news articles (20 Sports, 20 Politics, 20 Technology)")
print("✓ Train-Test Split: 75%-25% (stratified)")
print("✓ Preprocessing: Tokenization, stop word removal, stemming")
print("✓ TF-IDF: Manual implementation with cosine normalization")
print("✓ Naïve Bayes: Manual implementation with Laplace smoothing")
print("✓ Rocchio: Manual implementation with cosine similarity")
print("✓ Evaluation: Accuracy, Precision, Recall, F1-Score (all manual)")
print("✓ Comparison: Comprehensive performance analysis completed")

print("\n" + "="*80)
print("FINAL RESULTS")
print("="*80)

print(f"\nNaïve Bayes Classifier:")
print(f"  - Accuracy: {nb_metrics['accuracy']*100:.2f}%")
print(f"  - Macro F1-Score: {nb_metrics['macro_f1']:.4f}")

print(f"\nRocchio Classifier:")
print(f"  - Accuracy: {rocchio_metrics['accuracy']*100:.2f}%")
print(f"  - Macro F1-Score: {rocchio_metrics['macro_f1']:.4f}")

print("\n" + "="*80)
print("Assignment Complete! All outputs displayed.")
print("="*80)


ASSIGNMENT SUMMARY

✓ Dataset: 60 news articles (20 Sports, 20 Politics, 20 Technology)
✓ Train-Test Split: 75%-25% (stratified)
✓ Preprocessing: Tokenization, stop word removal, stemming
✓ TF-IDF: Manual implementation with cosine normalization
✓ Naïve Bayes: Manual implementation with Laplace smoothing
✓ Rocchio: Manual implementation with cosine similarity
✓ Evaluation: Accuracy, Precision, Recall, F1-Score (all manual)
✓ Comparison: Comprehensive performance analysis completed

FINAL RESULTS

Naïve Bayes Classifier:
  - Accuracy: 94.12%
  - Macro F1-Score: 0.9426

Rocchio Classifier:
  - Accuracy: 91.18%
  - Macro F1-Score: 0.9138

Assignment Complete! All outputs displayed.
